## VIATICOS CARGAR DATOS

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS corpac_viaticos;

-- =========================================
-- ELIMINAR TABLAS
-- =========================================
DROP TABLE IF EXISTS corpac_viaticos.fact_viaticos;
DROP TABLE IF EXISTS corpac_viaticos.dim_tiempo;
DROP TABLE IF EXISTS corpac_viaticos.dim_rubro;

-- =========================================
-- DIMENSION TIEMPO
-- =========================================
CREATE TABLE corpac_viaticos.dim_tiempo (

    id_tiempo BIGINT GENERATED ALWAYS AS IDENTITY,

    anio INT NOT NULL,
    trimestre INT NOT NULL,

    CONSTRAINT pk_dim_tiempo
    PRIMARY KEY (id_tiempo)

)
USING DELTA;

-- =========================================
-- DIMENSION RUBRO
-- =========================================
CREATE TABLE corpac_viaticos.dim_rubro (

    id_rubro BIGINT GENERATED ALWAYS AS IDENTITY,

    rubro STRING NOT NULL,

    CONSTRAINT pk_dim_rubro
    PRIMARY KEY (id_rubro)

)
USING DELTA;

-- =========================================
-- FACT TABLE
-- =========================================
CREATE TABLE corpac_viaticos.fact_viaticos (

    id_viatico BIGINT GENERATED ALWAYS AS IDENTITY,

    id_tiempo BIGINT NOT NULL,
    id_rubro BIGINT NOT NULL,

    monto DOUBLE,

    CONSTRAINT pk_fact_viaticos
    PRIMARY KEY (id_viatico),

    CONSTRAINT fk_fact_tiempo
    FOREIGN KEY (id_tiempo)
    REFERENCES corpac_viaticos.dim_tiempo(id_tiempo),

    CONSTRAINT fk_fact_rubro
    FOREIGN KEY (id_rubro)
    REFERENCES corpac_viaticos.dim_rubro(id_rubro)

)
USING DELTA;

In [0]:
# LEER CSV

ruta = "/Workspace/Users/dilan.flores.h@uni.pe/CORPAC/data_corpac/viaticos_consolidados.csv"

df = spark.read.csv(
    ruta,
    header=True,
    inferSchema=True
)

# CREAR VISTA TEMPORAL
df.createOrReplaceTempView("viaticos_csv")

display(df)

In [0]:
%sql

-- =========================================
-- INSERTAR TIEMPO
-- =========================================
INSERT INTO corpac_viaticos.dim_tiempo (
    anio,
    trimestre
)

SELECT DISTINCT
    TRY_CAST(ANIO AS INT),
    TRY_CAST(TRIMESTRE AS INT)

FROM viaticos_csv

WHERE TRY_CAST(ANIO AS INT) IS NOT NULL
AND TRY_CAST(TRIMESTRE AS INT) IS NOT NULL;

-- =========================================
-- INSERTAR RUBROS
-- =========================================
INSERT INTO corpac_viaticos.dim_rubro (
    rubro
)

SELECT DISTINCT
    RUBRO

FROM viaticos_csv

WHERE RUBRO IS NOT NULL
AND TRIM(RUBRO) <> '';

-- =========================================
-- INSERTAR FACT TABLE
-- =========================================
INSERT INTO corpac_viaticos.fact_viaticos (

    id_tiempo,
    id_rubro,
    monto
)

SELECT
    t.id_tiempo,
    r.id_rubro,

    TRY_CAST(v.MONTO AS DOUBLE)

FROM viaticos_csv v

JOIN corpac_viaticos.dim_tiempo t
ON TRY_CAST(v.ANIO AS INT) = t.anio
AND TRY_CAST(v.TRIMESTRE AS INT) = t.trimestre

JOIN corpac_viaticos.dim_rubro r
ON v.RUBRO = r.rubro;

## PENALIDADES LIMPIEZA

In [0]:
%pip install rapidfuzz --quiet

In [0]:
# MAGIC %pip install rapidfuzz

import pandas as pd
import re
from rapidfuzz import fuzz
import unicodedata

RUTA_CSV_ENTRADA = "/Workspace/Users/dilan.flores.h@uni.pe/CORPAC/data_corpac/penalidades_consolidadas.csv"
RUTA_CSV_SALIDA  = "/Workspace/Users/dilan.flores.h@uni.pe/CORPAC/data_corpac/penalidades_final.csv"
RUTA_REPORTE     = "/Workspace/Users/dilan.flores.h@uni.pe/CORPAC/data_corpac/reporte_unificaciones.csv"

UMBRAL_SIMILITUD = 85

# ==========================================
# CORRECCIONES MANUALES
# Si ves en el reporte que dos proveedores son
# el mismo pero el algoritmo no los unió,
# agrégalos aquí: "NOMBRE EN CSV" : "NOMBRE CORRECTO"
# ==========================================
CORRECCIONES_MANUALES = {
    # Ejemplos — edita con tus casos reales:
    # "CONSTRUCTORA RIOS S.A": "CONSTRUCTORA RIOS SA",
    # "CONSORCIO VIAL NORTE": "CONSORCIO VIAL NORTE SAC",
}

# ==========================================
# FUNCIONES DE NORMALIZACIÓN
# ==========================================
def quitar_acentos(t):
    return ''.join(c for c in unicodedata.normalize('NFD', t)
                   if unicodedata.category(c) != 'Mn')

def normalizar_base(texto):
    """Limpieza estándar: mayúsculas, sin puntos, sin caracteres especiales."""
    if pd.isna(texto) or str(texto).strip() == '':
        return ""
    t = str(texto).upper().strip()
    t = quitar_acentos(t)
    t = t.replace('.', '')           # S.A.C. → SAC
    t = re.sub(r'[-,;:()\'"*/#&]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    t = re.sub(r'[^\w\s]', '', t)
    return t

STOPWORDS = {
    'DE','LA','EL','LOS','LAS','DEL','Y','E','EN','A','THE',
    'SA','SAC','SCRL','SRL','EIRL','SL','CIA','CO','CORP',
    'CONTRATISTAS','CONTRATISTA','CONSTRUCTORA','SERVICIOS',
    'EMPRESA','PERU','PERUANA','PERUANO','SAS','LTDA','INC'
}

def normalizar_clave(texto):
    """Para comparar: sin stopwords, palabras ordenadas alfabéticamente."""
    base = normalizar_base(texto)
    palabras = sorted([p for p in base.split() if p not in STOPWORDS and len(p) > 1])
    return ' '.join(palabras)

# ==========================================
# CARGAR DATOS
# ==========================================
print("📂 Cargando CSV...")
df = pd.read_csv(RUTA_CSV_ENTRADA, dtype=str, encoding='utf-8-sig')
df['PROVEEDOR'] = df['PROVEEDOR'].fillna('').str.strip()

nombres_originales = [n for n in df['PROVEEDOR'].unique() if n.strip() != '']
print(f"   ➤ Proveedores únicos ANTES: {len(nombres_originales)}")

freq = df['PROVEEDOR'].value_counts().to_dict()

# ==========================================
# PASO 0: CORRECCIONES MANUALES
# Se aplican primero, antes de cualquier algoritmo
# ==========================================
mapa_final = {}
for n in nombres_originales:
    if n in CORRECCIONES_MANUALES:
        mapa_final[n] = CORRECCIONES_MANUALES[n]
    else:
        mapa_final[n] = n  # por defecto se mapea a sí mismo

# Actualizar nombres_originales con los ya corregidos
nombres_trabajo = list(set(mapa_final.values()))

# ==========================================
# PASO 1: AGRUPACIÓN EXACTA POR CLAVE
# Dos nombres con la misma clave normalizada
# son definitivamente el mismo proveedor
# ==========================================
print("🔑 Paso 1: agrupación por clave exacta...")

clave_a_canonico = {}   # clave_norm → nombre canónico (más frecuente)
nombre_a_clave   = {}   # nombre_original → clave_norm

for n in nombres_trabajo:
    clave = normalizar_base(n)   # usamos base (no la de stopwords) para exactitud
    nombre_a_clave[n] = clave
    if clave not in clave_a_canonico:
        clave_a_canonico[clave] = n
    else:
        # Gana el más frecuente
        actual = clave_a_canonico[clave]
        if freq.get(n, 0) > freq.get(actual, 0):
            clave_a_canonico[clave] = n

# Aplicar paso 1 al mapa
for n in nombres_trabajo:
    clave = nombre_a_clave[n]
    canonico_paso1 = clave_a_canonico[clave]
    # Actualizar solo si cambia
    original_key = next((k for k, v in mapa_final.items() if v == n), n)
    mapa_final[n] = canonico_paso1

# Recalcular nombres únicos tras paso 1
canonicos_paso1 = list(set(mapa_final.values()))
print(f"   ➤ Después del paso 1 (exacto): {len(canonicos_paso1)} únicos")

# ==========================================
# PASO 2: FUZZY MATCHING
# Solo entre los nombres que sobrevivieron
# el paso 1 (ya son distintos exactamente)
# ==========================================
print("🔍 Paso 2: fuzzy matching sobre residuos...")

# Preparar claves de comparación (con stopwords removidas y ordenadas)
claves_fuzzy   = [normalizar_clave(n) for n in canonicos_paso1]
procesados     = set()
mapa_fuzzy     = {}   # canonico_paso1 → canonico_final
clusters_fuzzy = []

for i, (clave_i, orig_i) in enumerate(zip(claves_fuzzy, canonicos_paso1)):
    if orig_i in procesados:
        continue
    cluster = [orig_i]
    procesados.add(orig_i)

    if len(clave_i) == 0:
        clusters_fuzzy.append(cluster)
        continue

    for j, (clave_j, orig_j) in enumerate(zip(claves_fuzzy, canonicos_paso1)):
        if i == j or orig_j in procesados:
            continue
        if len(clave_j) == 0:
            continue

        score1 = fuzz.ratio(clave_i, clave_j)
        score2 = fuzz.token_sort_ratio(clave_i, clave_j)
        score3 = fuzz.partial_ratio(clave_i, clave_j)
        score_final = max(score1, score2, score3)

        # Claves muy cortas: exigir más para evitar falsos positivos
        umbral = 96 if min(len(clave_i), len(clave_j)) < 6 else UMBRAL_SIMILITUD

        if score_final >= umbral:
            cluster.append(orig_j)
            procesados.add(orig_j)

    clusters_fuzzy.append(cluster)

for cluster in clusters_fuzzy:
    canonico = max(cluster, key=lambda n: freq.get(n, 0))
    for nombre in cluster:
        mapa_fuzzy[nombre] = canonico

# ==========================================
# COMBINAR PASOS: mapa_final + mapa_fuzzy
# ==========================================
for orig, tras_paso1 in mapa_final.items():
    mapa_final[orig] = mapa_fuzzy.get(tras_paso1, tras_paso1)

print(f"   ➤ Proveedores únicos DESPUÉS: {len(set(mapa_final.values()))}")

# ==========================================
# APLICAR AL DATAFRAME
# ==========================================
df['PROVEEDOR'] = df['PROVEEDOR'].map(lambda n: mapa_final.get(n, n))

# ==========================================
# REPORTE DE AUDITORÍA — EXPORTAR A CSV
# para que puedas revisar y agregar manuales
# ==========================================
cambios = [(orig, can) for orig, can in mapa_final.items() if orig != can]
cambios_df = pd.DataFrame(cambios, columns=['NOMBRE_ORIGINAL', 'NOMBRE_CANONICO'])
cambios_df = cambios_df.sort_values('NOMBRE_CANONICO').reset_index(drop=True)
cambios_df.to_csv(RUTA_REPORTE, index=False, encoding='utf-8-sig')

print(f"\n📋 Unificaciones realizadas: {len(cambios_df)}")
print(f"   ➤ Reporte guardado en: {RUTA_REPORTE}")
display(cambios_df)

# ==========================================
# ELIMINAR COLUMNA ARCHIVO
# ==========================================
if 'ARCHIVO' in df.columns:
    df = df.drop(columns=['ARCHIVO'])

# ==========================================
# GUARDAR CSV FINAL
# ==========================================
df.to_csv(RUTA_CSV_SALIDA, index=False, encoding='utf-8-sig')
print(f"\n✨ ¡ÉXITO! Guardado en: {RUTA_CSV_SALIDA}")
display(df)

## Exportar Datos

In [0]:
# MAGIC %pip install psycopg2-binary sqlalchemy

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS corpac_penalidades;
-- =========================================
-- 1. ELIMINAR TABLAS SI EXISTEN
-- =========================================
DROP TABLE IF EXISTS corpac_penalidades.fact_penalidad;
DROP TABLE IF EXISTS corpac_penalidades.dim_proveedor;
DROP TABLE IF EXISTS corpac_penalidades.dim_tiempo;

-- =========================================
-- 2. DIMENSIÓN PROVEEDOR
-- =========================================
CREATE TABLE corpac_penalidades.dim_proveedor (
    id_proveedor BIGINT GENERATED ALWAYS AS IDENTITY,
    proveedor STRING NOT NULL,
    ruc STRING,
    
    CONSTRAINT dim_proveedor_pk
    PRIMARY KEY (id_proveedor)
)
USING DELTA;

-- =========================================
-- 3. DIMENSIÓN TIEMPO
-- =========================================
CREATE TABLE corpac_penalidades.dim_tiempo (
    id_tiempo BIGINT GENERATED ALWAYS AS IDENTITY,
    anio INT NOT NULL,
    periodo STRING NOT NULL,

    CONSTRAINT dim_tiempo_pk
    PRIMARY KEY (id_tiempo)
)
USING DELTA;

-- =========================================
-- 4. TABLA FACT PENALIDAD
-- =========================================
CREATE TABLE corpac_penalidades.fact_penalidad (
    id_penalidad BIGINT GENERATED ALWAYS AS IDENTITY,

    id_proveedor BIGINT,
    id_tiempo BIGINT,

    monto_penalidad DOUBLE,
    monto_total_contrato DOUBLE,

    CONSTRAINT fk_fact_proveedor
    FOREIGN KEY (id_proveedor)
    REFERENCES corpac_penalidades.dim_proveedor(id_proveedor),

    CONSTRAINT fk_fact_tiempo
    FOREIGN KEY (id_tiempo)
    REFERENCES corpac_penalidades.dim_tiempo(id_tiempo)
)
USING DELTA;

In [0]:
# LEER CSV
ruta = "/Workspace/Users/dilan.flores.h@uni.pe/CORPAC/data_corpac/penalidades_final.csv"

df = spark.read.csv(
    ruta,
    header=True,
    inferSchema=True
)

# CREAR VISTA TEMPORAL
df.createOrReplaceTempView("penalidades_csv")

In [0]:
%sql

-- PROVEEDORES
INSERT INTO corpac_penalidades.dim_proveedor (proveedor, ruc)
SELECT DISTINCT
    PROVEEDOR,
    RUC
FROM penalidades_csv;

-- TIEMPO
INSERT INTO corpac_penalidades.dim_tiempo (anio, periodo)
SELECT DISTINCT
    ANIO,
    PERIODO
FROM penalidades_csv;

-- FACT TABLE
INSERT INTO corpac_penalidades.fact_penalidad (
    id_proveedor,
    id_tiempo,
    monto_penalidad,
    monto_total_contrato
)

SELECT
    p.id_proveedor,
    t.id_tiempo,
    c.MONTO_PENALIDAD,
    c.MONTO_TOTAL_CONTRATO

FROM penalidades_csv c

JOIN corpac_penalidades.dim_proveedor p
ON c.PROVEEDOR = p.proveedor
AND c.RUC = p.ruc

JOIN corpac_penalidades.dim_tiempo t
ON c.ANIO = t.anio
AND c.PERIODO = t.periodo;

In [0]:
%sql

SELECT *
FROM corpac_penalidades.fact_penalidad
LIMIT 10;

## ORDENES LIMPIEZA

In [0]:
# ==========================================
# NORMALIZAR Y UNIFICAR PROVEEDORES
# SOBRE ordenes_consolidadas.csv
# ==========================================

# MAGIC %pip install rapidfuzz

import pandas as pd
import re
import unicodedata
from rapidfuzz import fuzz

# ==========================================
# RUTAS
# ==========================================
RUTA_CSV_ENTRADA = "/Workspace/Users/dilan.flores.h@uni.pe/CORPAC/data_corpac/ordenes_consolidadas.csv"

RUTA_CSV_SALIDA = "/Workspace/Users/dilan.flores.h@uni.pe/CORPAC/data_corpac/ordenes_final.csv"

RUTA_REPORTE = "/Workspace/Users/dilan.flores.h@uni.pe/CORPAC/data_corpac/reporte_unificacion_proveedores.csv"

# ==========================================
# CONFIGURACIÓN
# ==========================================
UMBRAL_SIMILITUD = 88

# ==========================================
# CORRECCIONES MANUALES
# AGREGA AQUÍ LOS CASOS QUE QUIERAS FORZAR
# ==========================================
CORRECCIONES_MANUALES = {

    # EJEMPLOS:
    # "IBM DEL PERU S A C": "IBM PERU SAC",
    # "TELEFONICA DEL PERU SA": "TELEFONICA DEL PERU S.A.A.",
    # "CORPAC S A": "CORPAC SA"

}

# ==========================================
# FUNCIONES
# ==========================================

def quitar_acentos(texto):
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

def normalizar_base(texto):

    if pd.isna(texto):
        return ""

    t = str(texto).upper().strip()

    t = quitar_acentos(t)

    # quitar caracteres especiales
    t = t.replace('.', ' ')
    t = t.replace(',', ' ')
    t = t.replace('-', ' ')
    t = t.replace('/', ' ')
    t = t.replace('&', ' ')

    # quitar múltiples espacios
    t = re.sub(r'\s+', ' ', t)

    return t.strip()

STOPWORDS = {
    'SAC','SA','SAA','EIRL','SRL','SCRL','SAS',
    'EMPRESA','SERVICIOS','CONSTRUCTORA',
    'DEL','DE','LA','EL','LOS','LAS',
    'CORP','CORPORATION','PERU','PERUANA',
    'CIA','LTDA','INC','GROUP'
}

def normalizar_clave(texto):

    base = normalizar_base(texto)

    palabras = [
        p for p in base.split()
        if p not in STOPWORDS and len(p) > 1
    ]

    palabras = sorted(palabras)

    return ' '.join(palabras)

# ==========================================
# CARGAR CSV
# ==========================================
print("📂 Cargando consolidado...")

df = pd.read_csv(
    RUTA_CSV_ENTRADA,
    dtype=str,
    encoding='utf-8-sig'
)

df['PROVEEDOR'] = df['PROVEEDOR'].fillna('').str.strip()

# ==========================================
# FRECUENCIAS
# ==========================================
freq = df['PROVEEDOR'].value_counts().to_dict()

proveedores_originales = [
    p for p in df['PROVEEDOR'].unique()
    if p.strip() != ''
]

print(f"➤ Proveedores únicos ANTES: {len(proveedores_originales)}")

# ==========================================
# PASO 0 — MANUALES
# ==========================================
mapa_final = {}

for p in proveedores_originales:

    if p in CORRECCIONES_MANUALES:
        mapa_final[p] = CORRECCIONES_MANUALES[p]
    else:
        mapa_final[p] = p

proveedores_trabajo = list(set(mapa_final.values()))

# ==========================================
# PASO 1 — EXACT MATCH NORMALIZADO
# ==========================================
print("🔑 Paso 1: agrupación exacta...")

clave_a_canonico = {}

for p in proveedores_trabajo:

    clave = normalizar_base(p)

    if clave not in clave_a_canonico:
        clave_a_canonico[clave] = p

    else:
        actual = clave_a_canonico[clave]

        # gana el más frecuente
        if freq.get(p, 0) > freq.get(actual, 0):
            clave_a_canonico[clave] = p

for p in proveedores_trabajo:

    clave = normalizar_base(p)

    mapa_final[p] = clave_a_canonico[clave]

canonicos_paso1 = list(set(mapa_final.values()))

print(f"➤ Después del paso exacto: {len(canonicos_paso1)}")

# ==========================================
# PASO 2 — FUZZY MATCHING
# ==========================================
print("🔍 Paso 2: fuzzy matching...")

procesados = set()

mapa_fuzzy = {}

clusters = []

for i, p1 in enumerate(canonicos_paso1):

    if p1 in procesados:
        continue

    clave1 = normalizar_clave(p1)

    cluster = [p1]

    procesados.add(p1)

    for j, p2 in enumerate(canonicos_paso1):

        if i == j:
            continue

        if p2 in procesados:
            continue

        clave2 = normalizar_clave(p2)

        if clave1 == '' or clave2 == '':
            continue

        score1 = fuzz.ratio(clave1, clave2)
        score2 = fuzz.token_sort_ratio(clave1, clave2)
        score3 = fuzz.partial_ratio(clave1, clave2)

        score_final = max(score1, score2, score3)

        umbral = 95 if min(len(clave1), len(clave2)) < 6 else UMBRAL_SIMILITUD

        if score_final >= umbral:

            cluster.append(p2)

            procesados.add(p2)

    clusters.append(cluster)

# ==========================================
# CREAR MAPEO FINAL
# ==========================================
for cluster in clusters:

    # escoger el nombre más frecuente
    canonico = max(
        cluster,
        key=lambda x: freq.get(x, 0)
    )

    for nombre in cluster:
        mapa_fuzzy[nombre] = canonico

# combinar
for original, actual in mapa_final.items():

    mapa_final[original] = mapa_fuzzy.get(actual, actual)

# ==========================================
# APLICAR AL DATAFRAME
# ==========================================
df['PROVEEDOR_ORIGINAL'] = df['PROVEEDOR']

df['PROVEEDOR'] = df['PROVEEDOR'].map(
    lambda x: mapa_final.get(x, x)
)

# ==========================================
# REPORTE DE CAMBIOS
# ==========================================
cambios = [
    (orig, nuevo)
    for orig, nuevo in mapa_final.items()
    if orig != nuevo
]

reporte_df = pd.DataFrame(
    cambios,
    columns=['NOMBRE_ORIGINAL', 'NOMBRE_UNIFICADO']
)

reporte_df = reporte_df.sort_values(
    'NOMBRE_UNIFICADO'
).reset_index(drop=True)

reporte_df.to_csv(
    RUTA_REPORTE,
    index=False,
    encoding='utf-8-sig'
)
# ==========================================
# ELIMINAR COLUMNA ARCHIVO
# ==========================================
if 'ARCHIVO' in df.columns:
    df = df.drop(columns=['ARCHIVO'])

# ==========================================
# GUARDAR FINAL
# ==========================================
df.to_csv(
    RUTA_CSV_SALIDA,
    index=False,
    encoding='utf-8-sig'
)

# ==========================================
# RESULTADOS
# ==========================================
antes = len(proveedores_originales)
despues = len(set(df['PROVEEDOR']))

print("\n✅ UNIFICACIÓN COMPLETADA")
print(f"➤ Antes : {antes} proveedores únicos")
print(f"➤ Después: {despues} proveedores únicos")

print(f"\n📄 CSV FINAL:")
print(RUTA_CSV_SALIDA)

print(f"\n📋 REPORTE:")
print(RUTA_REPORTE)

display(reporte_df)
display(df)

## CARGAR DATOS

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS corpac_ordenes;

-- =========================================
-- ELIMINAR TABLAS
-- =========================================
DROP TABLE IF EXISTS corpac_ordenes.fact_orden;
DROP TABLE IF EXISTS corpac_ordenes.dim_proveedor;
DROP TABLE IF EXISTS corpac_ordenes.dim_tiempo;
DROP TABLE IF EXISTS corpac_ordenes.dim_origen;

-- =========================================
-- DIMENSION PROVEEDOR
-- =========================================
CREATE TABLE corpac_ordenes.dim_proveedor (
    
    id_proveedor BIGINT GENERATED ALWAYS AS IDENTITY,
    proveedor STRING NOT NULL,
    ruc STRING,

    CONSTRAINT pk_dim_proveedor
    PRIMARY KEY (id_proveedor)
)
USING DELTA;

-- =========================================
-- DIMENSION TIEMPO
-- =========================================
CREATE TABLE corpac_ordenes.dim_tiempo (

    id_tiempo BIGINT GENERATED ALWAYS AS IDENTITY,
    anio INT NOT NULL,
    periodo STRING NOT NULL,

    CONSTRAINT pk_dim_tiempo
    PRIMARY KEY (id_tiempo)
)
USING DELTA;

-- =========================================
-- DIMENSION ORIGEN
-- =========================================
CREATE TABLE corpac_ordenes.dim_origen (

    id_origen BIGINT GENERATED ALWAYS AS IDENTITY,
    origen STRING NOT NULL,

    CONSTRAINT pk_dim_origen
    PRIMARY KEY (id_origen)
)
USING DELTA;

-- =========================================
-- FACT TABLE
-- =========================================
CREATE TABLE corpac_ordenes.fact_orden (

    id_orden BIGINT GENERATED ALWAYS AS IDENTITY,

    id_proveedor BIGINT NOT NULL,
    id_tiempo BIGINT NOT NULL,
    id_origen BIGINT NOT NULL,

    nro_orden STRING,
    monto_soles DOUBLE,

    CONSTRAINT pk_fact_orden
    PRIMARY KEY (id_orden),

    CONSTRAINT fk_fact_proveedor
    FOREIGN KEY (id_proveedor)
    REFERENCES corpac_ordenes.dim_proveedor(id_proveedor),

    CONSTRAINT fk_fact_tiempo
    FOREIGN KEY (id_tiempo)
    REFERENCES corpac_ordenes.dim_tiempo(id_tiempo),

    CONSTRAINT fk_fact_origen
    FOREIGN KEY (id_origen)
    REFERENCES corpac_ordenes.dim_origen(id_origen)
)
USING DELTA;

In [0]:
ruta = "/Workspace/Users/dilan.flores.h@uni.pe/CORPAC/data_corpac/ordenes_final.csv"

df = spark.read.csv(
    ruta,
    header=True,
    inferSchema=True
)

# CREAR VISTA TEMPORAL
df.createOrReplaceTempView("ordenes_csv")

In [0]:
%sql

-- =========================================
-- INSERTAR PROVEEDORES
-- =========================================
INSERT INTO corpac_ordenes.dim_proveedor (
    proveedor,
    ruc
)

SELECT DISTINCT
    PROVEEDOR,
    RUC
FROM ordenes_csv
WHERE PROVEEDOR IS NOT NULL
AND TRIM(PROVEEDOR) <> '';

-- =========================================
-- INSERTAR TIEMPO
-- =========================================
INSERT INTO corpac_ordenes.dim_tiempo (
    anio,
    periodo
)

SELECT DISTINCT
    TRY_CAST(ANIO AS INT),
    PERIODO
FROM ordenes_csv
WHERE TRY_CAST(ANIO AS INT) IS NOT NULL;

-- =========================================
-- INSERTAR ORIGEN
-- =========================================
INSERT INTO corpac_ordenes.dim_origen (
    origen
)

SELECT DISTINCT
    ORIGEN
FROM ordenes_csv
WHERE ORIGEN IS NOT NULL
AND TRIM(ORIGEN) <> '';

-- =========================================
-- INSERTAR FACT TABLE
-- =========================================
INSERT INTO corpac_ordenes.fact_orden (
    id_proveedor,
    id_tiempo,
    id_origen,
    nro_orden,
    monto_soles
)

SELECT
    p.id_proveedor,
    t.id_tiempo,
    o.id_origen,

    c.NRO_ORDEN,
    TRY_CAST(c.MONTO_SOLES AS DOUBLE)

FROM ordenes_csv c

JOIN corpac_ordenes.dim_proveedor p
ON c.PROVEEDOR = p.proveedor
AND c.RUC = p.ruc

JOIN corpac_ordenes.dim_tiempo t
ON TRY_CAST(c.ANIO AS INT) = t.anio
AND c.PERIODO = t.periodo

JOIN corpac_ordenes.dim_origen o
ON c.ORIGEN = o.origen;

In [0]:
%sql
SELECT *
FROM corpac_ordenes.dim_tiempo;